# EgyptGPT Autoresearch

Autonomous AI research loop using Claude Code on your Claude subscription.
Based on Karpathy's [autoresearch](https://github.com/karpathy/autoresearch) pattern.

**Requirements:** Colab GPU runtime, Claude Pro/Max subscription.

## 1. Clone repo and install dependencies

In [ ]:
%cd /content
!rm -rf EgyptGPT
!git clone --recurse-submodules https://github.com/JLansey/EgyptGPT.git
%cd /content/EgyptGPT
!bash setup.sh

## 2. Mount Google Drive and prepare data

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

%cd /content/EgyptGPT

DATA_DIR = 'data/egypt_char'
DRIVE_DATA = '/content/drive/MyDrive/EgyptGPT_autoresearch/data'
DATA_FILES = ['train.bin', 'val.bin', 'meta.pkl']

# Try loading cached data from Drive
cached = all(os.path.exists(os.path.join(DRIVE_DATA, f)) for f in DATA_FILES)
if cached:
    print('Loading cached data from Google Drive...')
    for f in DATA_FILES:
        shutil.copy(os.path.join(DRIVE_DATA, f), os.path.join(DATA_DIR, f))
else:
    print('No cache. Running prepare.py...')
    # Must run as module import so hiero_transformer submodule is on the path
    !cd /content/EgyptGPT && python -c "from data.egypt_char import prepare"
    # Cache to Drive
    os.makedirs(DRIVE_DATA, exist_ok=True)
    for f in DATA_FILES:
        shutil.copy(os.path.join(DATA_DIR, f), os.path.join(DRIVE_DATA, f))
    print(f'Cached to {DRIVE_DATA}')

# Verify — fail hard if anything is missing
for f in DATA_FILES:
    path = os.path.join(DATA_DIR, f)
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'{path}: {os.path.getsize(path):,} bytes')
print('Data ready.')

## 3. Install Claude Code

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs tmux
!npm install -g @anthropic-ai/claude-code
!claude --version

## 4. Create non-root user and configure everything

`--dangerously-skip-permissions` is blocked under root (Colab default).
This creates a non-root user with full ownership of the repo, git config, Drive backup dir,
and pre-creates the autoresearch branch.

In [ ]:
import os
import subprocess
from google.colab import userdata
# Create non-root user and allow passwordless su
!useradd -m researcher 2>/dev/null || true
!passwd -d researcher

# Give researcher full ownership of the repo (including .git internals)
!chown -R researcher:researcher /content/EgyptGPT

# GPU access: open nvidia devices to all users
!chmod a+rw /dev/nvidia* /dev/dri/* 2>/dev/null || true

# Configure git for researcher
for cmd in [
    'git config --global --add safe.directory /content/EgyptGPT',
    'git config --global user.email autoresearch@colab',
    'git config --global user.name autoresearch',
]:
    subprocess.run(['sudo', '-u', 'researcher', 'bash', '-c', cmd], check=True)

DRIVE_ROOT = '/content/drive/MyDrive/EgyptGPT_autoresearch'
os.makedirs(DRIVE_ROOT, exist_ok=True)

try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception as e:
    raise RuntimeError('Missing Colab secret GITHUB_TOKEN. Add it in Colab Secrets before running this cell.') from e

if not github_token:
    raise RuntimeError('Colab secret GITHUB_TOKEN is empty. Set a valid GitHub token with repo push access.')

askpass_script = '/content/EgyptGPT/scripts/colab_git_askpass.sh'
sync_script = '/content/EgyptGPT/scripts/colab_sync_push.sh'
subprocess.run(['chmod', '+x', askpass_script, sync_script], check=True)

validate_push = subprocess.run([
    'sudo', '-u', 'researcher', 'env',
    'HOME=/home/researcher',
    'PATH=' + os.environ['PATH'],
    'GITHUB_TOKEN=' + github_token,
    'GIT_ASKPASS=' + askpass_script,
    'GIT_TERMINAL_PROMPT=0',
    'bash', '-lc', 'cd /content/EgyptGPT && git ls-remote --exit-code origin HEAD'
], capture_output=True, text=True)

if validate_push.returncode != 0:
    raise RuntimeError(
        'GITHUB_TOKEN could not authenticate to origin. '
        'Check that the token is valid and has repo push access.\n'
        f'stdout:\n{validate_push.stdout}\n'
        f'stderr:\n{validate_push.stderr}'
    )

# Smoke-test CUDA access for the researcher user using the preserved Colab GPU env
subprocess.run([
    'sudo', '-u', 'researcher', 'env',
    'HOME=/home/researcher',
    'PATH=/usr/local/nvidia/bin:' + subprocess.check_output(['bash', '-lc', 'printf %s "$PATH"'], text=True).strip(),
    'LD_LIBRARY_PATH=/usr/local/nvidia/lib:/usr/local/nvidia/lib64' + (
        ':' + subprocess.check_output(['bash', '-lc', 'printf %s "${LD_LIBRARY_PATH:-}"'], text=True).strip()
        if subprocess.check_output(['bash', '-lc', 'printf %s "${LD_LIBRARY_PATH:-}"'], text=True).strip()
        else ''
    ),
    'CUDA_VISIBLE_DEVICES=' + subprocess.check_output(['bash', '-lc', 'printf %s "${CUDA_VISIBLE_DEVICES:-0}"'], text=True).strip(),
    'bash', '--noprofile', '--norc', '-c',
    'python -c "import torch; print(\'CUDA:\', torch.cuda.is_available()); print(\'count:\', torch.cuda.device_count()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'no gpu\')"'
], check=False)

# Set Claude Code to use Opus model for researcher
!mkdir -p /home/researcher/.claude
!echo '{"model": "claude-opus-4-6"}' > /home/researcher/.claude/settings.json
!chown -R researcher:researcher /home/researcher/.claude

sync_proc = subprocess.Popen([
    'bash', sync_script
], env={
    **os.environ,
    'REPO_ROOT': '/content/EgyptGPT',
    'DRIVE_ROOT': DRIVE_ROOT,
    'SYNC_SECONDS': '150',
    'GITHUB_TOKEN': github_token,
    'GIT_ASKPASS': askpass_script,
})

print(f'Non-root user ready, git configured, sync/push loop running (pid {sync_proc.pid}).')
print(f'Drive sync root: {DRIVE_ROOT}')
print('Git pushes validated with Colab secret GITHUB_TOKEN.')

## 5. Log in to Claude (one-time, uses your subscription)

Open the **Colab terminal** (bottom-left terminal icon) and run:

```bash
cd /EgyptGPT && claude
```

It will show an OAuth URL. **Copy the URL, paste it in your browser, and log in.**
Once authenticated, type `/exit` to quit, then run the next cell.

In [ ]:
# Copy Claude credentials from root to the researcher user
!cp -r /root/.claude /home/researcher/.claude 2>/dev/null || true
# Re-apply Opus model setting (credential copy may overwrite settings)
!python3 /content/EgyptGPT/set_opus_model.py /home/researcher/.claude/settings.json
!chown -R researcher:researcher /home/researcher/.claude
print('Credentials copied, Opus model confirmed.')

## 6. Launch autoresearch

Open the **Colab terminal** and run:

```bash
tmux new-session -s autoresearch -d
```

Then enter the `researcher` shell with the GPU-preserving environment:

```bash
sudo -u researcher env HOME=/home/researcher PATH="/usr/local/nvidia/bin:$PATH" LD_LIBRARY_PATH="/usr/local/nvidia/lib:/usr/local/nvidia/lib64${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}" CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES:-0}" bash --noprofile --norc
```

Optional GPU check inside that shell:

```bash
python -c "import torch; print('CUDA:', torch.cuda.is_available()); print('count:', torch.cuda.device_count()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"
```

Then start Claude from `/content/EgyptGPT` with either model:

```bash
cd /content/EgyptGPT
claude --model claude-opus-4-6 --dangerously-skip-permissions
# or
claude --model claude-sonnet-4-6 --dangerously-skip-permissions
```

then paste in this prompt:
```
Read program.md in this directory. This is an autoresearch setup.
Data files are verified. Create or choose the autoresearch branch yourself as described in program.md.

1. Initialize results.tsv with the header row
2. Begin the experiment loop as described in program.md

Start with the baseline run, then iterate autonomously. Never stop.
```

`results.tsv` is synced to Google Drive every 150 seconds. The same loop also runs `git push` for the current branch every 150 seconds using the Colab secret `GITHUB_TOKEN`. Push output is appended to `EgyptGPT_autoresearch/git_push.log` in Drive.

**Detach:** `Ctrl+B` then `D` (keeps it running in background)

**Reattach:** `tmux attach -t autoresearch`

**Stop:** `tmux kill-session -t autoresearch`

## 7. Monitor

In [ ]:
!cat /content/EgyptGPT/results.tsv 2>/dev/null || echo 'No results yet.'

In [ ]:
!cd /content/EgyptGPT && git log --oneline -20

In [ ]:
!tmux has-session -t autoresearch 2>/dev/null && echo 'Running' || echo 'Stopped'